# TensorGuard quickstart — catch a shape bug before you run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/01_quickstart.ipynb)

Verify a model *statically* and *batch-polymorphically* from its source. The buggy variant below only raises at call time in stock PyTorch; TensorGuard flags it first, for every batch size at once.

In [ ]:
%pip install -q "git+https://github.com/thehalleyyoung/tensorguard.git"

In [ ]:
from tensorguard import verify_architecture

good = '''
import torch, torch.nn as nn
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))
'''
r = verify_architecture(good, input_shapes={'x': ('batch', 784)})
print('good model:', r.status, '| bugs:', len(r.bugs))
assert r.status == 'SAFE'

A one-line dimension typo (`256` → `255`) is caught:

In [ ]:
bad = good.replace('nn.Linear(256, 10)', 'nn.Linear(255, 10)')
r = verify_architecture(bad, input_shapes={'x': ('batch', 784)})
print('bad model:', r.status, '| bugs:', len(r.bugs))
assert r.status == 'UNSAFE'
print(r.bugs[0].message)